In [28]:
import re
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score,classification_report
from sklearn.naive_bayes import MultinomialNB,BernoulliNB
from sklearn.pipeline import Pipeline

In [29]:
nltk.download('stopwords',quiet=True)
stopw=set(stopwords.words('english'))
Stem=PorterStemmer()

In [30]:
def preprocess(text : str) -> str:
    if not isinstance(text,str) :
        return ""
    text=re.sub(r'<.*?>','',text)
    text=re.sub(r'[^a-zA-Z0-9\s]','',text.lower())
    tokens=[Stem.stem(w) for w in text.split() if w not in stopw]
    return ' '.join(tokens)


In [31]:
df=pd.read_csv('IMDBDataset.csv')
df.head(5)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [32]:
x=df['review']
y=df['sentiment'].replace({
    'positive' : 1 ,
    'negative' : 0

}).astype(int)

In [33]:
x=x.apply(preprocess)
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8)

In [42]:
pip=Pipeline([
    ('vect',CountVectorizer()),
    ('classifier',MultinomialNB())
])
param=[{
    'vect__max_features' : [4000, 8000, 5000],
    'vect__ngram_range': [(1, 1), (1, 2)],
    'classifier' : [MultinomialNB()],
    'classifier__alpha' : [0.1, 0.5, 1.0],
    'classifier__fit_prior' : [False,True]

},
{
    'vect__max_features' : [8000, 4000, 5000],
    'vect__ngram_range': [(1, 1), (1, 2)],
    'classifier' : [BernoulliNB()],
    'classifier__alpha' : [0.1, 0.5, 1.0],
    'classifier__fit_prior' : [False,True],
    'classifier__binarize': [0.0]

}]

In [43]:
gv=GridSearchCV(pip,param,cv=5,n_jobs=-1,scoring='accuracy')
gv.fit(xtrain,ytrain)
y_pred=gv.predict(xtest)
print("\nClassification Report:\n", classification_report(ytest, y_pred))


Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.84      0.85      4940
           1       0.85      0.88      0.86      5060

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000



In [44]:
print("\nAccuracy Report:\n", accuracy_score(ytest, y_pred))


Accuracy Report:
 0.8588


In [45]:
print(gv.best_params_)
print(gv.best_estimator_)

{'classifier': BernoulliNB(), 'classifier__alpha': 0.1, 'classifier__binarize': 0.0, 'classifier__fit_prior': False, 'vect__max_features': 8000, 'vect__ngram_range': (1, 2)}
Pipeline(steps=[('vect',
                 CountVectorizer(max_features=8000, ngram_range=(1, 2))),
                ('classifier', BernoulliNB(alpha=0.1, fit_prior=False))])
